# Week 1 — Dataset and Initial Data Preprocessing

Inventory all four medical CSV datasets and inspect their schema and quality. This phase does not train a model.


In [ ]:
from pathlib import Path
import re, numpy as np, pandas as pd, matplotlib.pyplot as plt
from urllib.request import urlretrieve

REPO_RAW_BASE = 'https://raw.githubusercontent.com/vyasanbmathew2008/projecttest/main/data/raw'
DATASET_FILES = [
    'heart_disease.csv',
    'diabetes_dataset.csv',
    'lung_disease_data.csv',
    'health_dataset.csv',
]
ROOT = Path.cwd()
if (ROOT / 'data/raw').exists():
    DATA_DIR = ROOT / 'data/raw'
elif (ROOT.parent / 'data/raw').exists():
    ROOT = ROOT.parent
    DATA_DIR = ROOT / 'data/raw'
else:
    ROOT = Path.cwd()
    DATA_DIR = ROOT / 'data/raw'
DATA_DIR.mkdir(parents=True, exist_ok=True)
for filename in DATASET_FILES:
    destination = DATA_DIR / filename
    if not destination.exists():
        url = f'{REPO_RAW_BASE}/{filename}'
        print(f'Downloading {filename} from {url}')
        urlretrieve(url, destination)

CSV_FILES = sorted(DATA_DIR.glob('*.csv'))
print('Project root:', ROOT.resolve())
print('CSV files:', [p.name for p in CSV_FILES])


In [ ]:
datasets = {}
for path in CSV_FILES:
    try:
        df = pd.read_csv(path, engine='python', on_bad_lines='skip')
    except TypeError:
        df = pd.read_csv(path, engine='python', on_bad_lines='skip')
    df.columns = [re.sub(r'\s+', ' ', str(c)).strip() for c in df.columns]
    datasets[path.stem] = df
    print(f'{path.name}: {len(df):,} rows x {len(df.columns)} columns')
rows = []
for name, df in datasets.items():
    rows.append({'dataset':name, 'rows':len(df), 'columns':len(df.columns), 'duplicates':int(df.duplicated().sum()), 'missing_cells':int(df.isna().sum().sum()), 'missing_%':round(df.isna().mean().mean()*100,2), 'numeric_columns':len(df.select_dtypes(include=np.number).columns), 'categorical_text_columns':len(df.select_dtypes(exclude=np.number).columns)})
profile_df = pd.DataFrame(rows).sort_values('rows', ascending=False)
display(profile_df)


In [ ]:
for name, df in datasets.items():
    print(f'\n### {name}')
    display(pd.DataFrame({'dtype':df.dtypes.astype(str), 'missing':df.isna().sum(), 'missing_%':(df.isna().mean()*100).round(2), 'unique':df.nunique(dropna=False)}).sort_values('missing_%', ascending=False).head(25))
fig, axes = plt.subplots(len(datasets), 2, figsize=(14, 4*len(datasets)))
if len(datasets) == 1: axes = np.array([axes])
for i, (name, df) in enumerate(datasets.items()):
    (df.isna().mean()*100).sort_values(ascending=False).head(12).plot.bar(ax=axes[i,0], title=f'{name}: missingness (%)')
    num = df.select_dtypes(include=np.number)
    if len(num.columns): num.iloc[:,:8].hist(ax=axes[i,1], bins=20)
    axes[i,1].set_title(f'{name}: numeric distributions')
plt.tight_layout()


In [ ]:
(ROOT/'artifacts').mkdir(exist_ok=True)
profile_df.to_csv(ROOT/'artifacts/week1_dataset_profile.csv', index=False)


## Exit criteria

Profile all four datasets independently. Use the target mapping in the Week 2 notebook; do not concatenate unrelated disease datasets.


---

# Week 2 — Complete Preprocessing and ML Modelling

Continue in this same notebook after reviewing the Week 1 profile. Run the Week 2 cells to train all four datasets and generate their pickle bundles.


# Week 2 — Complete Preprocessing and ML Modelling

Train one leakage-safe model pipeline per medical CSV dataset. The notebook compares logistic regression and random forest, then saves one metadata-rich `.pkl` bundle per dataset for `app.py`.\n\nThe training pipeline automatically excludes the post-outcome `Recovered` column from model inputs to prevent data leakage.

All datasets use the standardized target name `Disease`. Source target columns are mapped as: `Heart Disease Status` → `Disease`, `Target` → `Disease`, `Disease Type` → `Disease`.

In [ ]:
from pathlib import Path
import pickle
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, balanced_accuracy_score

from urllib.request import urlretrieve

REPO_RAW_BASE = 'https://raw.githubusercontent.com/vyasanbmathew2008/projecttest/main/data/raw'
DATASET_FILES = [
    'heart_disease.csv',
    'diabetes_dataset.csv',
    'lung_disease_data.csv',
    'health_dataset.csv',
]
ROOT = Path.cwd()
if (ROOT / 'data/raw').exists():
    DATA_DIR = ROOT / 'data/raw'
elif (ROOT.parent / 'data/raw').exists():
    ROOT = ROOT.parent
    DATA_DIR = ROOT / 'data/raw'
else:
    ROOT = Path.cwd()
    DATA_DIR = ROOT / 'data/raw'
DATA_DIR.mkdir(parents=True, exist_ok=True)
for filename in DATASET_FILES:
    destination = DATA_DIR / filename
    if not destination.exists():
        url = f'{REPO_RAW_BASE}/{filename}'
        print(f'Downloading {filename} from {url}')
        urlretrieve(url, destination)

DATASET_CONFIG = {
    'heart_disease': {'file': 'heart_disease.csv', 'target_aliases': ['Disease', 'Heart Disease Status']},
    'diabetes_dataset': {'file': 'diabetes_dataset.csv', 'target_aliases': ['Disease', 'Target']},
    'lung_disease_data': {'file': 'lung_disease_data.csv', 'target_aliases': ['Disease', 'Disease Type']},
    'health_dataset': {'file': 'health_dataset.csv', 'target_aliases': ['Disease']},
}
TARGET_COLUMN = 'Disease'
RANDOM_STATE = 42
EXCLUDED_FEATURE_COLUMNS = {'recovered'}

def read_table(path):
    # The uploaded health CSV has a few rows with extra symptom fields; skip malformed rows, and remove rows with missing targets before training.
    return pd.read_csv(path, engine='python', on_bad_lines='skip')

def clean_columns(df):
    result = df.copy()
    result.columns = [re.sub(r'\s+', ' ', str(c)).strip() for c in result.columns]
    return result.drop_duplicates()

print('Datasets:', ', '.join(DATASET_CONFIG))


In [ ]:
model_dir = ROOT / 'models'
model_dir.mkdir(parents=True, exist_ok=True)
all_results = []

for dataset_name, config in DATASET_CONFIG.items():
    print(f'\n===== {dataset_name} =====')
    df = clean_columns(read_table(DATA_DIR / config['file']))
    target = next((column for column in config['target_aliases'] if column in df.columns), None)
    if target is None:
        raise ValueError(
            f'No disease target found in {config["file"]}; '
            f'expected one of {config["target_aliases"]}; columns: {list(df.columns)}'
        )
    if target != TARGET_COLUMN:
        df = df.rename(columns={target: TARGET_COLUMN})
    target = TARGET_COLUMN

    if dataset_name == 'heart_disease':
        df[TARGET_COLUMN] = (
            df[TARGET_COLUMN]
            .astype('string')
            .str.strip()
            .str.lower()
            .map({
                'yes': 'Heart Disease',
                'no': 'No Heart Disease',
                '1': 'Heart Disease',
                '0': 'No Heart Disease',
            })
            .fillna(df[TARGET_COLUMN].astype('string').str.strip())
        )

    X = df.drop(columns=[target])
    y = df[target].astype(str).str.strip()
    valid_target = df[target].notna()
    X, y = X.loc[valid_target].reset_index(drop=True), y.loc[valid_target].reset_index(drop=True)
    if y.nunique() < 2:
        raise ValueError(f'{dataset_name} needs at least two target classes')
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, stratify=y, random_state=RANDOM_STATE)
    numeric = X_train.select_dtypes(include=np.number).columns.tolist()
    categorical = [c for c in X_train.columns if c not in numeric]
    preprocessor = ColumnTransformer([
        ('numeric', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric),
        ('categorical', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical),
    ])
    candidates = {
        'logistic_regression': LogisticRegression(max_iter=1500, class_weight='balanced', random_state=RANDOM_STATE),
        'random_forest': RandomForestClassifier(n_estimators=50, max_depth=12, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE),
    }
    results, fitted = [], {}
    for model_name, estimator in candidates.items():
        pipeline = Pipeline([('preprocessor', preprocessor), ('model', estimator)])
        pipeline.fit(X_train, y_train)
        predicted = pipeline.predict(X_test)
        score = balanced_accuracy_score(y_test, predicted)
        results.append({'dataset': dataset_name, 'model': model_name, 'balanced_accuracy': score})
        fitted[model_name] = pipeline
        print(f'{model_name}: balanced accuracy={score:.4f}')
        print(classification_report(y_test, predicted, zero_division=0))
    results_df = pd.DataFrame(results).sort_values('balanced_accuracy', ascending=False)
    all_results.extend(results)
    best_name = str(results_df.iloc[0]['model'])
    feature_schema = {}
    for column in X.columns:
        series = X[column]
        if column in numeric:
            numeric_values = pd.to_numeric(series, errors='coerce').dropna()
            feature_schema[column] = {'kind': 'numeric', 'default': float(numeric_values.median()) if len(numeric_values) else 0.0, 'min': float(numeric_values.min()) if len(numeric_values) else -1e6, 'max': float(numeric_values.max()) if len(numeric_values) else 1e6}
        else:
            options = [str(value) for value in series.dropna().astype(str).unique()[:100]]
            feature_schema[column] = {'kind': 'categorical', 'options': options or ['']}
    bundle = {
        'pipeline': fitted[best_name],
        'model_name': best_name,
        'dataset_name': dataset_name,
        'target': target,
        'columns': X.columns.tolist(),
        'classes': sorted(y.unique().tolist()),
        'feature_schema': feature_schema,
        'scores': {str(row['model']): float(row['balanced_accuracy']) for _, row in results_df.iterrows()},
    }
    model_path = model_dir / f'{dataset_name}.pkl'
    with model_path.open('wb') as handle:
        pickle.dump(bundle, handle)
    print('Saved:', model_path)

results_path = ROOT / 'artifacts/week2_model_results.csv'
pd.DataFrame(all_results).to_csv(results_path, index=False)
print('Saved results:', results_path)


## Exit criteria

Confirm each target is appropriate, review class imbalance and metrics, verify four `.pkl` bundles in `models/`, and run `streamlit run app.py`.